In [1]:
import pandas as pd
import pickle
import numpy as np

In [2]:
df = pickle.load(open('dataset_level2.pkl','rb'))

In [3]:
df.head()

# batting_team
# bowling team
# city
# current score
# ball left
# wickets left
# current rr
# last five

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,venue
0,2,Australia,Sri Lanka,0.1,0,0,NaN,Melbourne Cricket Ground
1,2,Australia,Sri Lanka,0.2,0,0,NaN,Melbourne Cricket Ground
2,2,Australia,Sri Lanka,0.3,1,0,NaN,Melbourne Cricket Ground
3,2,Australia,Sri Lanka,0.4,2,0,NaN,Melbourne Cricket Ground
4,2,Australia,Sri Lanka,0.5,0,0,NaN,Melbourne Cricket Ground


In [4]:
df.isnull().sum()

match_id               0
batting_team           0
bowling_team           0
ball                   0
runs                   0
player_dismissed       0
city                8549
venue                  0
dtype: int64

In [5]:
df[df['city'].isnull()]['venue'].value_counts()

venue
Dubai International Cricket Stadium        3092
Pallekele International Cricket Stadium    2066
Melbourne Cricket Ground                   1453
Sydney Cricket Ground                       749
Adelaide Oval                               498
Harare Sports Club                          372
Sylhet International Cricket Stadium        128
Sharjah Cricket Stadium                     127
Carrara Oval                                 64
Name: count, dtype: int64

In [6]:
cities = np.where(df['city'].isnull(),df['venue'].str.split().apply(lambda x:x[0]),df['city'])

In [7]:
df['city'] = cities

In [8]:
df.isnull().sum()

match_id            0
batting_team        0
bowling_team        0
ball                0
runs                0
player_dismissed    0
city                0
venue               0
dtype: int64

In [9]:
df.drop(columns=['venue'],inplace=True)

In [10]:
df['city']

0         Melbourne
1         Melbourne
2         Melbourne
3         Melbourne
4         Melbourne
            ...    
349231      Colombo
349232      Colombo
349233      Colombo
349234      Colombo
349235      Colombo
Name: city, Length: 102183, dtype: object

In [11]:
eligible_cities = df['city'].value_counts()[df['city'].value_counts() > 600].index.tolist()

# df['city'].value_counts()

# df['city'].value_counts()[df['city'].value_counts() > 6]

In [12]:
df = df[df['city'].isin(eligible_cities)]

In [13]:
df.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne


In [23]:
# Convert runs to numeric (fix dtype)
df['runs'] = pd.to_numeric(df['runs'], errors='coerce').fillna(0)

# Calculate cumulative score per match
df['current_score'] = df.groupby('match_id')['runs'].cumsum()


# df['current_score'] = df.groupby('match_id').cumsum()['runs']

In [24]:
df.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,over,ball_no,balls_bowled,balls_left,current_score
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,1,1,119,0
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,2,2,118,0
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,0,3,3,117,1
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,0,4,4,116,3
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,0,5,5,115,3


In [25]:
df['over'] = df['ball'].apply(lambda x:str(x).split(".")[0])
df['ball_no'] = df['ball'].apply(lambda x:str(x).split(".")[1])
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,over,ball_no,balls_bowled,balls_left,current_score
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,1,1,119,0
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,2,2,118,0
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,0,3,3,117,1
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,0,4,4,116,3
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,0,5,5,115,3
...,...,...,...,...,...,...,...,...,...,...,...,...
349231,2900,Sri Lanka,Australia,19.3,1,0,Colombo,19,3,117,3,125
349232,2900,Sri Lanka,Australia,19.4,0,0,Colombo,19,4,118,2,125
349233,2900,Sri Lanka,Australia,19.5,0,1,Colombo,19,5,119,1,125
349234,2900,Sri Lanka,Australia,19.6,2,0,Colombo,19,6,120,0,127


In [26]:
df['balls_bowled'] = (df['over'].astype('int')*6) + df['ball_no'].astype('int')
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,over,ball_no,balls_bowled,balls_left,current_score
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,1,1,119,0
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,2,2,118,0
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,0,3,3,117,1
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,0,4,4,116,3
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,0,5,5,115,3
...,...,...,...,...,...,...,...,...,...,...,...,...
349231,2900,Sri Lanka,Australia,19.3,1,0,Colombo,19,3,117,3,125
349232,2900,Sri Lanka,Australia,19.4,0,0,Colombo,19,4,118,2,125
349233,2900,Sri Lanka,Australia,19.5,0,1,Colombo,19,5,119,1,125
349234,2900,Sri Lanka,Australia,19.6,2,0,Colombo,19,6,120,0,127


In [27]:
df['balls_left'] = 120 - df['balls_bowled']
df['balls_left'] = df['balls_left'].apply(lambda x:0 if x<0 else x)
df.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,over,ball_no,balls_bowled,balls_left,current_score
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,1,1,119,0
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,2,2,118,0
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,0,3,3,117,1
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,0,4,4,116,3
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,0,5,5,115,3


In [28]:
# df['player_dismissed'] = df['player_dismissed'].apply(lambda x:0 if x=='0' else 1)
# df['player_dismissed'] = df['player_dismissed'].astype('int')
# df['player_dismissed'] = df.groupby('match_id').cumsum()['player_dismissed']
# df['wickets_left'] = 10 - df['player_dismissed']

df['player_dismissed'] = df['player_dismissed'].apply(lambda x: 0 if x=='0' else 1).astype(int)

df['player_dismissed'] = df.groupby('match_id')['player_dismissed'].cumsum()

df['wickets_left'] = 10 - df['player_dismissed']


In [29]:
df.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,over,ball_no,balls_bowled,balls_left,current_score,wickets_left
0,2,Australia,Sri Lanka,0.1,0,1,Melbourne,0,1,1,119,0,9
1,2,Australia,Sri Lanka,0.2,0,2,Melbourne,0,2,2,118,0,8
2,2,Australia,Sri Lanka,0.3,1,3,Melbourne,0,3,3,117,1,7
3,2,Australia,Sri Lanka,0.4,2,4,Melbourne,0,4,4,116,3,6
4,2,Australia,Sri Lanka,0.5,0,5,Melbourne,0,5,5,115,3,5


In [30]:
df['crr'] = (df['current_score']*6)/df['balls_bowled']

In [31]:
df.head()

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,over,ball_no,balls_bowled,balls_left,current_score,wickets_left,crr
0,2,Australia,Sri Lanka,0.1,0,1,Melbourne,0,1,1,119,0,9,0.0
1,2,Australia,Sri Lanka,0.2,0,2,Melbourne,0,2,2,118,0,8,0.0
2,2,Australia,Sri Lanka,0.3,1,3,Melbourne,0,3,3,117,1,7,2.0
3,2,Australia,Sri Lanka,0.4,2,4,Melbourne,0,4,4,116,3,6,4.5
4,2,Australia,Sri Lanka,0.5,0,5,Melbourne,0,5,5,115,3,5,3.6


In [33]:
# FIX runs column
df['runs'] = pd.to_numeric(df['runs'], errors='coerce').fillna(0).astype(int)

# group by match_id
groups = df.groupby('match_id')
match_ids = df['match_id'].unique()
last_five = []

# rolling feature
for mid in match_ids:
    run_series = groups.get_group(mid)['runs']
    last_five.extend(run_series.rolling(window=30).sum().tolist())


# groups = df.groupby('match_id')

# match_ids = df['match_id'].unique()
# last_five = []
# for id in match_ids:
#     last_five.extend(groups.get_group(id).rolling(window=30).sum()['runs'].values.tolist())

In [34]:
df['last_five'] = last_five

In [38]:
# df[['batting_team','bowling_team','city','current_score','balls_left','wickets_left','crr','last_five','']]

In [36]:
final_df = df.groupby('match_id').sum()['runs'].reset_index().merge(df,on='match_id')

In [37]:
final_df=final_df[['batting_team','bowling_team','city','current_score','balls_left','wickets_left','crr','last_five','runs_x']]

In [39]:
final_df.dropna(inplace=True)

In [40]:
final_df.isnull().sum()

batting_team     0
bowling_team     0
city             0
current_score    0
balls_left       0
wickets_left     0
crr              0
last_five        0
runs_x           0
dtype: int64

In [41]:
final_df = final_df.sample(final_df.shape[0])

In [42]:
final_df.sample(2)

,batting_team,bowling_team,city,current_score,balls_left,wickets_left,crr,last_five,runs_x
43637,West Indies,South Africa,Kingston,62,77,-33,8.651163,46.0,207
85388,Sri Lanka,West Indies,Bangalore,94,18,-95,5.529412,29.0,122


In [43]:
X = final_df.drop(columns=['runs_x'])
y = final_df['runs_x']
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [44]:
X_train.head()

,batting_team,bowling_team,city,current_score,balls_left,wickets_left,crr,last_five
53499,Sri Lanka,Bangladesh,Johannesburg,70,56,-55,6.562500,31.0
72299,India,England,Mumbai,91,50,-66,7.800000,27.0
15297,Pakistan,England,Manchester,64,75,-36,8.533333,42.0
20817,India,Sri Lanka,Colombo,79,59,-52,7.770492,37.0
21946,West Indies,Australia,Gros Islet,72,61,-53,7.322034,28.0


In [45]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score,mean_absolute_error

In [48]:
trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'),
     ['batting_team', 'bowling_team', 'city'])
],
remainder='passthrough')


# trf = ColumnTransformer([
#     ('trf',OneHotEncoder(sparse_output==False,drop='first'),['batting_team','bowling_team','city'])
# ]
# ,remainder='passthrough')

In [49]:
pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',StandardScaler()),
    ('step3',XGBRegressor(n_estimators=1000,learning_rate=0.2,max_depth=12,random_state=1))
])

In [50]:
pipe.fit(X_train,y_train)
y_pred = pipe.predict(X_test)
print(r2_score(y_test,y_pred))
print(mean_absolute_error(y_test,y_pred))

0.975131630897522
2.2880263328552246


In [51]:
pickle.dump(pipe,open('pipe.pkl','wb'))

In [73]:
import xgboost
xgboost.__version__

'1.4.2'